**1. Setup and Imports**

In [238]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from collections import deque
from typing import Any, Callable, Dict, Iterable, List, Optional, Tuple
import heapq
import itertools
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches


**2. Foundations from Part A (Provided)**

In [239]:
class Problem(ABC):
    """Abstract base class for a search problem."""

    @abstractmethod
    def initial_state(self) -> Any:
        """Return the start state."""
        pass

    @abstractmethod
    def is_goal(self, state: Any) -> bool:
        """Return True if state is a goal state."""
        pass

    @abstractmethod
    def actions(self, state: Any) -> List[Any]:
        """Return the legal actions available in the given state."""
        pass

    @abstractmethod
    def result(self, state: Any, action: Any) -> Any:
        """Return the next state after applying action in state."""
        pass

    @abstractmethod
    def action_cost(self, state: Any, action: Any, next_state: Any) -> float:
        """Return the cost of applying action in state to reach next_state."""
        pass


In [240]:
@dataclass
class Node:
    state: Any
    parent: Optional["Node"] = None
    action: Optional[Any] = None
    path_cost: float = 0
    depth: int = 0

    def __post_init__(self):
        if self.parent is not None:
            self.depth = self.parent.depth + 1


@dataclass
class SearchResult:
    algorithm: str
    status: str
    solution: Optional[Node]
    nodes_expanded: int
    max_frontier_size: int
    reached_count: int = 0
    limit: Optional[int] = None
    iterations: Optional[List[Dict[str, Any]]] = None

    @property
    def path(self) -> Optional[List[Any]]:
        if self.solution is None:
            return None
        return reconstruct_path(self.solution)
    
    @property
    def solution_depth(self) -> Optional[int]:
        if self.solution is None:
            return None
        return self.solution.depth

    @property
    def solution_cost(self) -> Optional[float]:
        if self.solution is None:
            return None
        return self.solution.path_cost


In [241]:
def reconstruct_path(node: Node) -> List[Any]:
    """Return the list of states from the root node to this node."""
    path = []
    while node is not None:
        path.append(node.state)
        node = node.parent
    path.reverse()
    return path


def reconstruct_actions(node: Node) -> List[Any]:
    """Return the list of actions from the root node to this node."""
    actions = []
    while node is not None and node.parent is not None:
        actions.append(node.action)
        node = node.parent
    actions.reverse()
    return actions


def result_to_row(result: SearchResult) -> Dict[str, Any]:
    """Convert a SearchResult object into a row for a pandas DataFrame."""
    return {
        "Algorithm": result.algorithm,
        "Status": result.status,
        "Solution depth": result.solution_depth,
        "Solution cost": result.solution_cost,
        "Nodes expanded": result.nodes_expanded,
        "Max frontier": result.max_frontier_size,
        "Reached states": result.reached_count,
    }

def show_results(results: List[SearchResult]) -> pd.DataFrame:
    """Display results as a DataFrame."""
    return pd.DataFrame([result_to_row(r) for r in results])

**TO DOs IN 2**

In [242]:
class Problem(ABC):
    """Abstract base class for a search problem."""

    @abstractmethod
    def initial_state(self) -> Any:
        pass

    @abstractmethod
    def is_goal(self, state: Any) -> bool:
        pass

    @abstractmethod
    def actions(self, state: Any) -> List[Any]:
        pass

    @abstractmethod
    def result(self, state: Any, action: Any) -> Any:
        pass

    @abstractmethod
    def action_cost(self, state: Any, action: Any, next_state: Any) -> float:
        pass


@dataclass
class Node:
    state: Any
    parent: Optional["Node"] = None
    action: Optional[Any] = None
    path_cost: float = 0.0
    depth: int = 0

    def __post_init__(self):
        if self.parent is not None:
            self.depth = self.parent.depth + 1


@dataclass
class SearchResult:
    algorithm: str
    status: str
    solution: Optional[Node]
    nodes_expanded: int
    max_frontier_size: int
    reached_count: int = 0
    limit: Optional[int] = None
    iterations: Optional[List[Dict[str, Any]]] = None

    @property
    def path(self) -> Optional[List[Any]]:
        if self.solution is None:
            return None
        return reconstruct_path(self.solution)

    @property
    def solution_depth(self) -> Optional[int]:
        if self.solution is None:
            return None
        return self.solution.depth

    @property
    def solution_cost(self) -> Optional[float]:
        if self.solution is None:
            return None
        return self.solution.path_cost


def reconstruct_path(node: Node) -> List[Any]:
    path = []
    while node is not None:
        path.append(node.state)
        node = node.parent
    path.reverse()
    return path


def result_to_row(result: SearchResult) -> Dict[str, Any]:
    return {
        "Algorithm": result.algorithm,
        "Status": result.status,
        "Solution depth": result.solution_depth,
        "Solution cost": result.solution_cost,
        "Nodes expanded": result.nodes_expanded,
        "Max frontier": result.max_frontier_size,
        "Reached states": result.reached_count,
    }


def show_results(results: List[SearchResult]) -> pd.DataFrame:
    return pd.DataFrame([result_to_row(r) for r in results])


MOVES = {
    "UP": (-1, 0),
    "DOWN": (1, 0),
    "LEFT": (0, -1),
    "RIGHT": (0, 1),
}


class GridProblem(Problem):
    def __init__(
        self, grid: List[List[int]], start: Tuple[int, int], goal: Tuple[int, int]
    ):
        self.grid = grid
        self.start = start
        self.goal = goal
        self.rows = len(grid)
        self.cols = len(grid[0])

    def initial_state(self) -> Tuple[int, int]:
        return self.start
    # TODO1: Implement the is_goal method to check if the current state is the goal state.
    def is_goal(self, state: Tuple[int, int]) -> bool:
        return state == self.goal

    def in_bounds(self, state: Tuple[int, int]) -> bool:
        row, col = state
        return 0 <= row < self.rows and 0 <= col < self.cols

    def is_free(self, state: Tuple[int, int]) -> bool:
        row, col = state
        return self.grid[row][col] == 0

    def actions(self, state: Tuple[int, int]) -> List[str]:
        # TODO2: Implement the actions method to return a list of legal actions from the current state.
        # create a list to hold the legal actions
        legal_actions = []
        # loop through the possible moves and check if they are legal
        for action, (dr, dc) in MOVES.items():
            # calculate the neighbor state
            neighbor = (state[0] + dr, state[1] + dc)
            # check if the neighbor state is within bounds and free
            if self.in_bounds(neighbor) and self.is_free(neighbor):
                # add the action to the list of legal actions
                legal_actions.append(action)
        # return the list of legal actions
        return legal_actions

    def result(self, state: Tuple[int, int], action: str) -> Tuple[int, int]:
        # TODO3: Implement the result method to return the next state after applying an action to the current state.
        # calculate the next state based on the action
        dr, dc = MOVES[action]
        # return the next state as a tuple of (row, col)
        return (state[0] + dr, state[1] + dc)

    def action_cost(
        self, state: Tuple[int, int], action: str, next_state: Tuple[int, int]
    ) -> float:
        # TODO4: Implement the action_cost method to return the cost of applying an action in the current state to reach the next state.
        # return a constant cost of 1.0 for each action
        return 1.0


class SearchAlgorithm(ABC):
    """Base class for search algorithms with proper indentation context."""

    def expand(self, problem: Problem, node: Node) -> Iterable[Node]:
        # TODO 5:
        # Implement the AIMA-style EXPAND(problem, node).
        s = node.state
        for action in problem.actions(s):
            s_prime = problem.result(s, action)
            cost = node.path_cost + problem.action_cost(s, action, s_prime)
            yield Node(state=s_prime, parent=node, action=action, path_cost=cost)

    @abstractmethod
    def search(self, problem: Problem) -> SearchResult:
        pass


In [243]:
def plot_path(grid: List[List[int]], path: List[Tuple[int, int]], start: Tuple[int, int], goal: Tuple[int, int]) -> None:
    """Plot the grid and the path from start to goal."""
    rows, cols = len(grid), len(grid[0])
    fig, ax = plt.subplots(figsize=(cols, rows))
    ax.set_xlim(0, cols)
    ax.set_ylim(0, rows)
    ax.set_xticks(np.arange(0, cols + 1, 1))
    ax.set_yticks(np.arange(0, rows + 1, 1))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.grid(True)

    # Draw the grid
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 1:
                rect = patches.Rectangle((c, rows - r - 1), 1, 1, linewidth=1, edgecolor='black', facecolor='black')
                ax.add_patch(rect)

    # Draw the path
    if path:
        path_x = [c + 0.5 for r, c in path]
        path_y = [rows - r - 0.5 for r, c in path]
        ax.plot(path_x, path_y, color='blue', linewidth=2)

    # Mark start and goal
    ax.text(start[1] + 0.5, rows - start[0] - 0.5, 'S', color='green', fontsize=12, ha='center', va='center')
    ax.text(goal[1] + 0.5, rows - goal[0] - 0.5, 'G', color='red', fontsize=12, ha='center', va='center')

    plt.show() 

**3. Weighted Terrain: WeightedGridProblem**

In [244]:
class WeightedGridProblem(GridProblem):
    """A grid problem where entering a cell costs that cell's terrain cost."""

    def __init__(
        self,
        grid: List[List[int]],
        start: Tuple[int, int],
        goal: Tuple[int, int],
        terrain_costs: List[List[float]],
    ):
        """
        grid:
            2D list where 0 = free cell and 1 = obstacle.

        terrain_costs:
            2D list, same shape as grid. terrain_costs[r][c] is the cost of
            ENTERING cell (r, c). All values must be >= 1.
        """
        super().__init__(grid, start, goal)
        self.terrain_costs = terrain_costs

    def action_cost(
        self,
        state: Tuple[int, int],
        action: str,
        next_state: Tuple[int, int],
    ) -> float:
        """Return the cost of entering the next_state."""
        row, col = next_state
        return self.terrain_costs[row][col]
    

**3.1 Self-Check for WeightedGridProblem**

In [245]:
wtest_grid = [
    [0, 0, 0],
    [0, 0, 0],
    [0, 0, 0],
]

wtest_costs = [
    [1, 7, 1],
    [1, 1, 1],
    [1, 1, 1],
]

wtest_problem = WeightedGridProblem(
    wtest_grid, start=(0, 0), goal=(2, 2), terrain_costs=wtest_costs
)

# Entering the windy cell (0, 1) costs 7.
assert wtest_problem.action_cost((0, 0), "RIGHT", (0, 1)) == 7
# Entering the calm cell (1, 0) costs 1.
assert wtest_problem.action_cost((0, 0), "DOWN", (1, 0)) == 1
# Everything inherited from GridProblem still works.
assert wtest_problem.actions((0, 0)) == ["DOWN", "RIGHT"]
assert wtest_problem.is_goal((2, 2)) is True

print("WeightedGridProblem self-check passed.")


WeightedGridProblem self-check passed.


**4. Heuristic Functions**

In [246]:
def manhattan_distance(state: Tuple[int, int], goal: Tuple[int, int]) -> float:
    row1, col1 = state
    row2, col2 = goal
    return float(abs(row1 - row2) + abs(col1 - col2))


def euclidean_distance(state: Tuple[int, int], goal: Tuple[int, int]) -> float:
    return float(math.dist(state, goal))


def zero_heuristic(state: Tuple[int, int], goal: Tuple[int, int]) -> float:
    return 0.0


**4.1 Admissibility and Consistency on Our Grid**

In [247]:
def manhattan_distance(state: Tuple[int, int], goal: Tuple[int, int]) -> float:
     
    # TODO 2:
    #1. Return the Manhattan distance between state and goal.
    row1, col1 = state
    row2, col2 = goal
    # 2. Return abs(row1 - row2) + abs(col1 - col2).
    return float(abs(row1 - row2) + abs(col1 - col2))       
    raise NotImplementedError("Complete manhattan_distance")


def euclidean_distance(state: Tuple[int, int], goal: Tuple[int, int]) -> float:
    row1, col1 = state
    row2, col2 = goal
    # TODO 3:
    # Return the straight-line distance between state and goal.
    return float(math.sqrt((row1 - row2) ** 2 + (col1 - col2) ** 2))
    # Hint: math.sqrt(...) or math.dist(state, goal).
    raise NotImplementedError("Complete euclidean_distance")


def zero_heuristic(state: Tuple[int, int], goal: Tuple[int, int]) -> float:
    """h(n) = 0 for every node. Provided.

    A* with the zero heuristic degenerates into Uniform-Cost Search —
    admissible, consistent, and completely uninformative.
    """
    return 0.0
    

**4.2 Self-Check for the Heuristics**

In [248]:
assert manhattan_distance((0, 0), (2, 2)) == 4
assert manhattan_distance((3, 5), (3, 5)) == 0
assert abs(euclidean_distance((0, 0), (3, 4)) - 5.0) < 1e-9
assert euclidean_distance((1, 1), (1, 1)) == 0
assert zero_heuristic((0, 0), (9, 9)) == 0

# Euclidean never exceeds Manhattan (so Manhattan dominates Euclidean).
for s in [(0, 0), (2, 7), (5, 1)]:
    assert euclidean_distance(s, (9, 9)) <= manhattan_distance(s, (9, 9)) + 1e-9

print("Heuristic self-check passed.")


Heuristic self-check passed.


**5. The Priority-Queue Frontier (Provided)**

In [249]:
class PriorityQueue:
    """A min-priority queue of (priority, node) pairs built on heapq."""

    def __init__(self):
        self._heap: List[Tuple[float, int, Node]] = []
        self._counter = itertools.count()

    def push(self, priority: float, node: Node) -> None:
        heapq.heappush(self._heap, (priority, next(self._counter), node))

    def pop(self) -> Node:
        """Remove and return the node with the LOWEST priority."""
        priority, count, node = heapq.heappop(self._heap)
        return node

    def __len__(self) -> int:
        return len(self._heap)

    def __bool__(self) -> bool:
        return len(self._heap) > 0


**6. The BestFirstSearch Framework**

In [250]:
class BestFirstSearch(SearchAlgorithm):
    """Generic best-first search. Subclasses define the evaluation function f(n)."""

    algorithm_name = "BestFirst"

    def __init__(
        self,
        heuristic: Callable[[Tuple[int, int], Tuple[int, int]], float] = zero_heuristic,
    ):
        self.heuristic = heuristic

    def h(self, node: Node, problem: Problem) -> float:
        """Heuristic estimate from this node's state to the goal."""
        return self.heuristic(node.state, problem.goal)

    def evaluation(self, node: Node, problem: Problem) -> float:
        """f(n). Subclasses override this single method."""
        raise NotImplementedError("Subclasses must define evaluation(node, problem)")

    def search(self, problem: Problem) -> SearchResult:
        """Perform best-first search on the given problem."""
        # TODO 4:
        # Steps:
        # 1. Create the initial node from problem.initial_state().
        start_node = Node(state=problem.initial_state())

        # 2. Create a PriorityQueue frontier and push the initial node
        #    with priority self.evaluation(node, problem).
        frontier = PriorityQueue()
        frontier.push(self.evaluation(start_node, problem), start_node)

        # 3. Create a reached DICTIONARY mapping state -> Node, containing
        #    the initial state.
        reached = {start_node.state: start_node}

        # 4. Initialise counters: nodes_expanded = 0, max_frontier_size = 1.
        nodes_expanded = 0
        max_frontier_size = 1

        # 5. While the frontier is not empty:
        while frontier:
            # a. pop the node with the lowest f-value.
            node = frontier.pop()

            # b. if problem.is_goal(node.state): return a SearchResult with
            #    status "success" (use self.algorithm_name, nodes_expanded,
            #    max_frontier_size, and len(reached)).
            if problem.is_goal(node.state):
                return SearchResult(
                    algorithm=self.algorithm_name,
                    status="success",
                    solution=node,
                    nodes_expanded=nodes_expanded,
                    max_frontier_size=max_frontier_size,
                    reached_count=len(reached),
                )

            # c. increment nodes_expanded.
            nodes_expanded += 1

            # d. loop over each child in self.expand(problem, node):
            for child in self.expand(problem, node):
                # i. s = child.state
                s = child.state

                # ii. if s not in reached OR
                #     child.path_cost < reached[s].path_cost:
                if s not in reached or child.path_cost < reached[s].path_cost:
                    # reached[s] = child
                    reached[s] = child
                    # push child with priority self.evaluation(child, problem)
                    frontier.push(self.evaluation(child, problem), child)

            # e. update max_frontier_size with len(frontier).
            if len(frontier) > max_frontier_size:
                max_frontier_size = len(frontier)

        # 6. If the loop ends, return a SearchResult with status, failure
        # and solution=None.
        return SearchResult(
            algorithm=self.algorithm_name,
            status="failure",
            solution=None,
            nodes_expanded=nodes_expanded,
            max_frontier_size=max_frontier_size,
            reached_count=len(reached),
        )

**7. Greedy Best-First Search**

In [251]:
class GreedyBestFirstSearch(BestFirstSearch):
    # TODO 5:
    algorithm_name = "Greedy"
    # Greedy Best-First Search: f(n) = h(n).
    def evaluation(self, node: Node, problem: Problem) -> float:
        return self.h(node, problem)


8. A* Search

In [252]:
class AStarSearch(BestFirstSearch):
    algorithm_name = "A*"

    def evaluation(self, node: Node, problem: Problem) -> float:
        # TODO 6:
        # A* search: f(n) = g(n) + h(n).
        n = node.path_cost + self.h(node, problem)
        return n

        
        
    
        raise NotImplementedError("Complete AStarSearch.evaluation")


**9. Uniform-Cost Search as a Special Case (Provided)**

In [253]:
class UniformCostSearch(BestFirstSearch):
    algorithm_name = "UCS"

    def __init__(self):
        super().__init__(heuristic=zero_heuristic)

    def evaluation(self, node: Node, problem: Problem) -> float:
        return node.path_cost


10. Weighted A* Search

In [254]:
class WeightedAStarSearch(BestFirstSearch):
    def __init__(self, heuristic, weight: float = 2.0):
        super().__init__(heuristic=heuristic)
        self.weight = weight
        self.algorithm_name = f"Weighted A* (W={weight})"

    def evaluation(self, node: Node, problem: Problem) -> float:
        # TODO 7:
        # Weighted A*: f(n) = g(n) + W * h(n).
        n =node.path_cost + (self.weight * self.h(node, problem))
        return n


**10.1 Self-Check for the Algorithms**

In [255]:
check_grid = [[0, 0, 0], [1, 1, 0], [0, 0, 0]]
check_problem = GridProblem(check_grid, start=(0, 0), goal=(2, 2))

for algo in [
    GreedyBestFirstSearch(manhattan_distance),
    AStarSearch(manhattan_distance),
    WeightedAStarSearch(manhattan_distance, weight=2),
    UniformCostSearch(),
]:
    res = algo.search(check_problem)
    assert res.status == "success"
    assert res.solution_cost == 4.0

print("Algorithm self-check passed perfectly.")


Algorithm self-check passed perfectly.


**11.Run the Algorithms on the Part A Sample Map**

In [256]:
sample_grid = [
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [1, 1, 1, 0, 1, 0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 1, 1, 1, 0, 1, 0, 1, 1],
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 1, 1, 1, 1, 1, 0],
    [0, 0, 1, 0, 0, 0, 0, 0, 1, 0],
    [1, 0, 1, 1, 1, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 1, 1, 1, 0, 0, 0, 1, 0],
]
start, goal = (0, 0), (9, 9)
uniform_problem = GridProblem(sample_grid, start, goal)

uniform_results = [
    UniformCostSearch().search(uniform_problem),
    AStarSearch(manhattan_distance).search(uniform_problem),
    WeightedAStarSearch(manhattan_distance, weight=2).search(uniform_problem),
    GreedyBestFirstSearch(manhattan_distance).search(uniform_problem),
]

display(show_results(uniform_results))


,Algorithm,Status,Solution depth,Solution cost,Nodes expanded,Max frontier,Reached states
0,UCS,success,18,18.0,52,5,56
1,A*,success,18,18.0,18,5,23
2,Weighted A* (W=2),success,18,18.0,18,5,23
3,Greedy,success,18,18.0,18,5,23


In [257]:


# Print the metrics evaluation table
print("--- Uniform Map Performance Metrics ---")
display(show_results(uniform_results))

if uniform_results[1].path:
    plot_path(
        sample_grid,
        start,
        goal,
        path=uniform_results[1].path,
        title="A* Solution Path (uniform costs)",
    )

if uniform_results[3].path:
    plot_path(
        sample_grid,
        start,
        goal,
        path=uniform_results[3].path,
        title="Greedy Best-First Solution Path (uniform costs)",
    )


--- Uniform Map Performance Metrics ---


,Algorithm,Status,Solution depth,Solution cost,Nodes expanded,Max frontier,Reached states
0,UCS,success,18,18.0,52,5,56
1,A*,success,18,18.0,18,5,23
2,Weighted A* (W=2),success,18,18.0,18,5,23
3,Greedy,success,18,18.0,18,5,23


TypeError: plot_path() got multiple values for argument 'path'

**12. The Turbulence Map: Where Greedy Goes Wrong**

In [258]:
weighted_grid = [
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 1, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
]

terrain_costs = [
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 7, 7, 7, 7, 1, 1, 1, 1],
    [1, 1, 7, 7, 7, 7, 1, 1, 1, 1],
    [1, 1, 3, 3, 3, 3, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
]

w_problem = WeightedGridProblem(weighted_grid, (0, 0), (5, 9), terrain_costs)

weighted_results = [
    UniformCostSearch().search(w_problem),
    AStarSearch(manhattan_distance).search(w_problem),
    WeightedAStarSearch(manhattan_distance, weight=2.5).search(w_problem),
    GreedyBestFirstSearch(manhattan_distance).search(w_problem),
]

display(show_results(weighted_results))


,Algorithm,Status,Solution depth,Solution cost,Nodes expanded,Max frontier,Reached states
0,UCS,success,14,14.0,47,10,52
1,A*,success,14,14.0,40,11,48
2,Weighted A* (W=2.5),success,14,14.0,14,13,27
3,Greedy,success,14,14.0,14,13,27


In [ ]:
weighted_grid = [
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 1, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
]

terrain_costs = [
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 7, 7, 7, 7, 1, 1, 1, 1],
    [1, 1, 7, 7, 7, 7, 1, 1, 1, 1],
    [1, 1, 3, 3, 3, 3, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
]

w_problem = WeightedGridProblem(weighted_grid, (0, 0), (5, 9), terrain_costs)

weighted_results = [
    UniformCostSearch().search(w_problem),
    AStarSearch(manhattan_distance).search(w_problem),
    WeightedAStarSearch(manhattan_distance, weight=2.5).search(w_problem),
    GreedyBestFirstSearch(manhattan_distance).search(w_problem),
]

display(show_results(weighted_results))
# Plot the solution paths for A* Search and Greedy Best-First Search (if they found a solution).
if weighted_results[1].path:
    plot_path(
        weighted_grid,
        start,
        goal,
        path=weighted_results[1].path,
        title="A* Solution Path (weighted costs)",
    )                                                                               
    
if weighted_results[3].path:
    plot_path(
        weighted_grid,
        start,
        goal,
        path=weighted_results[3].path,
        title="Greedy Best-First Solution Path (weighted costs)",
    )       
    
    

,Algorithm,Status,Solution depth,Solution cost,Nodes expanded,Max frontier,Reached states
0,UCS,success,14,14.0,47,10,52
1,A*,success,14,14.0,40,11,48
2,Weighted A* (W=2.5),success,14,14.0,14,13,27
3,Greedy,success,14,14.0,14,13,27


TypeError: plot_path() got multiple values for argument 'path'

In [259]:
# Display A* Path Strategy
if weighted_results[1].path:
    plot_path(
        weighted_grid,
        (0, 0),
        (5, 9),
        path=weighted_results[1].path,
        terrain_costs=terrain_costs,
        title="A* Optimal Route (Safely Routes Around Storm Water)",
    )

# Display Greedy Path Strategy
if weighted_results[3].path:
    plot_path(
        weighted_grid,
        (0, 0),
        (5, 9),
        path=weighted_results[3].path,
        terrain_costs=terrain_costs,
        title="Greedy Suboptimal Route (Trapped in High-Drain Zone)",
    )


TypeError: plot_path() got multiple values for argument 'path'